# Selected Subject Plots, Fig 3C Style

This notebook plots selected state-grid subjects in the visual style used by `notebooks/for_nips/nips_figures.ipynb` Fig 3C:

- accuracy comparison: human vs model, multi-subject grid
- oral vs model true-hypothesis probability: one SVG per subject
- SVG output with transparent background


In [19]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import yaml
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.Bayesian_state.utils.oral_process import Oral_center_analysis, Oral_region_analysis

font_path = PROJECT_ROOT / "src" / "Arial.ttf"
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    prop = font_manager.FontProperties(fname=str(font_path))
    mpl.rcParams["font.family"] = prop.get_name()

mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["axes.linewidth"] = 2


## Configuration

Edit this cell for the subjects/config/result directory you want to plot.

In [30]:
# Example: condition 1 subjects 125 and 126.
SUBJECT_IDS = [230]
SUBJECT_LABELS = [230]

# One or more result directories containing subject_<id>.json.
RESULT_DIRS = [
    PROJECT_ROOT / "results" / "state-based-grid-result" / "pmh" / "cond2",
]

# One or more grid optimization YAMLs. The notebook uses oral.mode and oral data paths from these.
CONFIG_PATHS = [
    PROJECT_ROOT / "configs" / "grid_opt_cfg" / "pmh_cond2.yaml",
]

EVAL_PREDICTION_MODE = "posterior_t_minus_1"
MODEL_LABEL = "PMH"

OUTPUT_DIR = PROJECT_ROOT / "results" / "selected_subject"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fig 3C styling.
COLOR_MODEL_ACC = "#B38665"
COLOR_MODEL_K = "#20978B"
COLOR_HUMAN = "#A6A6A6"
XTICK_INTERVAL = 128
SEGMENT_INTERVAL = 64

# Fig 3C historically pads 15 empty points before sliding accuracy.
# Set to None to align from inferred window size instead.
ACCURACY_PADDING = 15

# Fig 3C k/oral comparison uses a 16-trial rolling window.
K_WINDOW_SIZE = 16

# Model curve in oral_vs_model:
# - "topk_hit": rolling hit rate for whether target k appears in model posterior top-k.
# - "posterior": rolling posterior probability of target k.
MODEL_K_CURVE = "posterior"

# None follows oral default: condition 1 -> top 4, condition 2/3 -> top 10.
MODEL_TOP_K = None

ACC_FIGSIZE = (8, 7)
K_FIGSIZE = (3, 1.7)
NROW_ACC = 2


In [10]:
def add_segmentation_lines(ax, max_trial, interval=64, **line_kwargs):
    for x in range(interval, int(max_trial) + 1, interval):
        ax.axvline(x=x, **line_kwargs)


def load_yaml(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}


def resolve_relative(base_file: Path, raw_path: str | Path) -> Path:
    path = Path(raw_path)
    if not path.is_absolute():
        path = (base_file.parent / path).resolve()
    return path


def find_subject_json(subject_id: int, result_dirs: list[Path]) -> Path:
    matches = []
    for result_dir in result_dirs:
        candidate = result_dir / f"subject_{subject_id}.json"
        if candidate.exists():
            matches.append(candidate)
    if not matches:
        raise FileNotFoundError(f"subject_{subject_id}.json not found in {result_dirs}")
    if len(matches) > 1:
        print(f"Warning: multiple result files for subject {subject_id}; using {matches[0]}")
    return matches[0]


def load_subject_payload(subject_id: int, result_dirs: list[Path]) -> dict[str, Any]:
    path = find_subject_json(subject_id, result_dirs)
    with path.open("r", encoding="utf-8") as f:
        payload = json.load(f)
    payload["_json_path"] = str(path)
    return payload


def get_eval_metrics(payload: dict[str, Any], eval_mode: str) -> dict[str, Any]:
    metrics_by_mode = payload.get("metrics_by_mode") or {}
    if eval_mode not in metrics_by_mode:
        raise KeyError(
            f"subject_{payload.get('subject_id')} lacks metrics_by_mode[{eval_mode!r}]. "
            f"Available: {sorted(metrics_by_mode)}"
        )
    return metrics_by_mode[eval_mode]


def infer_window_size(metrics: dict[str, Any]) -> int:
    true_acc = metrics.get("true_acc") or []
    sliding = metrics.get("sliding_pred_acc") or []
    if true_acc and sliding:
        return int(len(true_acc) - len(sliding))
    return 16


def build_plot_results(subject_ids: list[int]) -> tuple[dict[int, dict[str, Any]], dict[int, dict[str, Any]]]:
    acc_results = {}
    model_results = {}
    for sid in subject_ids:
        payload = load_subject_payload(sid, RESULT_DIRS)
        metrics = get_eval_metrics(payload, EVAL_PREDICTION_MODE)
        win = infer_window_size(metrics)
        acc_results[sid] = {
            "condition": payload.get("condition"),
            "sliding_true_acc": metrics.get("sliding_true_acc"),
            "sliding_pred_acc": metrics.get("sliding_pred_acc"),
            "sliding_pred_acc_std": metrics.get("sliding_pred_acc_std"),
            "window_size": win,
            "n_trials": len(metrics.get("true_acc") or []),
            "json_path": payload["_json_path"],
        }
        model_results[sid] = {
            "condition": payload.get("condition"),
            "step_results": payload.get("best_step_results") or payload.get("step_results") or [],
            "json_path": payload["_json_path"],
        }
    return acc_results, model_results


In [11]:
def load_oral_hits(config_paths: list[Path]) -> dict[int, dict[str, Any]]:
    oral_hits: dict[int, dict[str, Any]] = {}
    for config_path in config_paths:
        cfg = load_yaml(config_path)
        oral_cfg = cfg.get("oral") or {}
        mode = str(oral_cfg.get("mode", "center")).strip().lower()
        if mode not in {"center", "region"}:
            raise ValueError(f"Unsupported oral.mode={mode!r} in {config_path}")

        data_key = f"{mode}_data_path"
        if data_key not in oral_cfg:
            raise KeyError(f"Missing oral.{data_key} in {config_path}")
        data_path = resolve_relative(config_path, oral_cfg[data_key])
        oral_df = pd.read_csv(data_path)

        if mode == "center":
            hits = Oral_center_analysis().get_oral_hypo_hits(oral_df, window_size=K_WINDOW_SIZE)
        else:
            n_samples = int(oral_cfg.get("region_n_samples", 1000))
            hits = Oral_region_analysis().get_oral_hypo_hits(
                oral_df,
                window_size=K_WINDOW_SIZE,
                n_samples=n_samples,
            )

        overlap = set(oral_hits).intersection(hits)
        if overlap:
            print(f"Warning: oral hits overwritten for subjects {sorted(overlap)}")
        oral_hits.update(hits)
        print(f"Loaded oral {mode} hits from {data_path}")
    return oral_hits


def rolling_nanmean(values: list[float] | np.ndarray, window_size: int) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    rolling = []
    for idx in range(len(values)):
        if idx + 1 < window_size:
            rolling.append(np.nan)
            continue
        window = values[idx - window_size + 1 : idx + 1]
        if np.all(np.isnan(window)):
            rolling.append(np.nan)
        else:
            rolling.append(float(np.nanmean(window)))
    return np.asarray(rolling, dtype=float)


def recompute_oral_rolling_hits(oral_hits: dict[int, dict[str, Any]], window_size: int) -> dict[int, dict[str, Any]]:
    out = {}
    for sid, odict in oral_hits.items():
        hits = np.asarray(odict.get("hits", []), dtype=float)
        rolling = rolling_nanmean(hits, window_size)
        copied = dict(odict)
        copied["rolling_hits"] = rolling.tolist()
        out[sid] = copied
    return out


In [12]:
def _get_post_max(hypo_details: dict[Any, Any], k: int) -> float:
    entry = hypo_details.get(k)
    if entry is None:
        entry = hypo_details.get(str(k))
    if not isinstance(entry, dict):
        return 0.0
    try:
        return float(entry.get("post_max", 0.0))
    except (TypeError, ValueError):
        return 0.0


def resolve_top_k(condition: int, top_k: int | None = None) -> int:
    if top_k is not None and top_k > 0:
        return int(top_k)
    return 4 if int(condition) == 1 else 10


def iter_hypo_posteriors(hypo_details: dict[Any, Any]) -> list[tuple[int, float]]:
    values = []
    if not isinstance(hypo_details, dict):
        return values
    for key, entry in hypo_details.items():
        try:
            hypo_idx = int(key)
        except (TypeError, ValueError):
            continue
        if not isinstance(entry, dict):
            continue
        try:
            post = float(entry.get("post_max", 0.0))
        except (TypeError, ValueError):
            post = 0.0
        values.append((hypo_idx, post))
    return values


def extract_model_k_rolling(results_sub: dict[str, Any], k_special: int, win: int) -> np.ndarray:
    post_vals = []
    for step in results_sub.get("step_results", []):
        post_vals.append(_get_post_max(step.get("hypo_details", {}), k_special))
    return pd.Series(post_vals, dtype=float).rolling(window=win, min_periods=win).mean().to_numpy()


def extract_model_topk_hit_rolling(
    results_sub: dict[str, Any],
    k_special: int,
    condition: int,
    win: int,
    top_k: int | None = None,
) -> np.ndarray:
    resolved_top_k = resolve_top_k(condition, top_k)
    hits = []
    for step in results_sub.get("step_results", []):
        values = iter_hypo_posteriors(step.get("hypo_details", {}))
        if not values:
            hits.append(np.nan)
            continue
        values.sort(key=lambda item: item[1], reverse=True)
        top_hypos = {hypo_idx for hypo_idx, _ in values[:resolved_top_k]}
        hits.append(1.0 if int(k_special) in top_hypos else 0.0)
    return rolling_nanmean(hits, win)


def extract_model_oral_curve(
    results_sub: dict[str, Any],
    k_special: int,
    condition: int,
    win: int,
) -> np.ndarray:
    if MODEL_K_CURVE == "posterior":
        return extract_model_k_rolling(results_sub, k_special, win)
    if MODEL_K_CURVE == "topk_hit":
        return extract_model_topk_hit_rolling(results_sub, k_special, condition, win, MODEL_TOP_K)
    raise ValueError(f"Unsupported MODEL_K_CURVE={MODEL_K_CURVE!r}")


def plot_accuracy_fig3c_style(
    results: dict[int, dict[str, Any]],
    subject_ids: list[int],
    subject_labels: list[int | str],
    nrow: int,
    save_path: Path,
    figsize: tuple[float, float] = (8, 7),
):
    n_sub = len(subject_ids)
    ncol = int(np.ceil(n_sub / nrow))
    fig, axes = plt.subplots(nrow, ncol, figsize=figsize, sharex=False, sharey=False)
    axes_flat = np.array(axes).reshape(-1)
    plt.tight_layout(h_pad=3)

    for idx, subject_id in enumerate(subject_ids):
        ax = axes_flat[idx]
        info = results[subject_id]
        pad_n = int(ACCURACY_PADDING) if ACCURACY_PADDING is not None else int(info.get("window_size", 16)) - 1
        padding = [np.nan] * max(pad_n, 0)

        true_acc = padding + list(info["sliding_true_acc"])
        pred = padding + list(info["sliding_pred_acc"])
        std = padding + list(info["sliding_pred_acc_std"])
        x_vals = np.arange(1, len(true_acc) + 1)

        ax.plot(x_vals, true_acc, label="Human", color=COLOR_HUMAN, linewidth=2)
        ax.plot(x_vals, pred, color=COLOR_MODEL_ACC, linewidth=2, clip_on=False)
        low = np.asarray(pred, dtype=float) - np.asarray(std, dtype=float)
        high = np.asarray(pred, dtype=float) + np.asarray(std, dtype=float)
        ax.fill_between(x_vals, low, high, color=COLOR_MODEL_ACC, alpha=0.3)

        add_segmentation_lines(
            ax,
            len(x_vals),
            interval=SEGMENT_INTERVAL,
            color="grey",
            alpha=0.3,
            linestyle="dashed",
            linewidth=1,
        )

        ax.set_xlim(0, len(x_vals) + 1)
        ax.set_ylim(0, 1)

        col = idx % ncol
        yticks = np.arange(0, 1.01, 0.2)
        ax.set_yticks(yticks)
        if col == 0:
            ax.set_yticklabels([f"{y:.1f}" for y in yticks], fontsize=16)
        else:
            ax.set_yticklabels([])
            ax.set_ylabel(None)

        xticks = range(0, int(ax.get_xlim()[1]) + 1, XTICK_INTERVAL)
        ax.set_xticks(xticks)
        ax.set_xticklabels(xticks, fontsize=16)
        ax.set_xlabel(None)

        ax.set_facecolor("none")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        for spine in ["left", "bottom"]:
            ax.spines[spine].set_linewidth(2)

        ax.text(
            0.07,
            0.87,
            f"S{subject_labels[idx]}",
            transform=ax.transAxes,
            fontsize=20,
            ha="left",
            va="bottom",
            color="black",
        )

    fig.text(0.5, -0.03, "Trial", ha="center", fontsize=19)
    fig.text(-0.035, 0.5, "Learning performance", va="center", rotation="vertical", fontsize=19)

    for j in range(n_sub, nrow * ncol):
        axes_flat[j].axis("off")

    fig.savefig(save_path, bbox_inches="tight", transparent=True)
    print(f"Figure saved to {save_path}")
    plt.close(fig)


def plot_oral_vs_model_fig3c_style(
    oral_hypo_hits: dict[int, dict[str, Any]],
    results: dict[int, dict[str, Any]],
    subject_id: int,
    subject_label: int | str,
    save_path: Path,
    figsize: tuple[float, float] = (3, 1.7),
):
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor("none")

    odict = oral_hypo_hits[subject_id]
    rolling_hits = odict["rolling_hits"]
    n_steps = len(rolling_hits)
    x_vals = np.arange(1, n_steps + 1)

    ax.plot(x_vals, rolling_hits, lw=2, label="Human", color=COLOR_HUMAN, clip_on=False)

    condition = int(odict["condition"])
    k_special = 0 if condition == 1 else 42
    rolling_k = extract_model_oral_curve(results[subject_id], k_special, condition, K_WINDOW_SIZE)
    ax.plot(x_vals[K_WINDOW_SIZE - 1:], rolling_k[K_WINDOW_SIZE - 1:], lw=2, color=COLOR_MODEL_K, clip_on=False)

    ax.set_xlim(0, len(x_vals) + 1)
    ax.set_ylim(0, 1)

    yticks = np.arange(0, 1.01, 0.2)
    ax.set_yticks(yticks)
    ax.set_yticklabels([])
    ax.set_ylabel(None)

    xticks = range(0, int(ax.get_xlim()[1]) + 1, XTICK_INTERVAL)
    ax.set_xticks(xticks)
    ax.set_xticklabels([])
    ax.set_xlabel(None)

    ax.set_facecolor("none")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_linewidth(2)

    fig.savefig(save_path, bbox_inches="tight", transparent=True)
    print(f"Figure saved to {save_path}")
    plt.close(fig)


## Load And Plot

In [31]:
accuracy_results, model_results = build_plot_results(SUBJECT_IDS)
oral_hypo_hits = load_oral_hits(CONFIG_PATHS)
oral_hypo_hits = recompute_oral_rolling_hits(oral_hypo_hits, K_WINDOW_SIZE)

for sid in SUBJECT_IDS:
    if sid not in oral_hypo_hits:
        raise KeyError(f"subject {sid} not found in oral hits")

print("Loaded subjects:")
for sid in SUBJECT_IDS:
    print(sid, "<-", accuracy_results[sid]["json_path"])


Loaded oral center hits from /home/yangjiong/CategoryLearning_gitcode/data/processed/Task2_processed.csv
Loaded subjects:
230 <- /home/yangjiong/CategoryLearning_gitcode/results/state-based-grid-result/pmh/cond2/subject_230.json


In [32]:
plot_accuracy_fig3c_style(
    accuracy_results,
    subject_ids=SUBJECT_IDS,
    subject_labels=SUBJECT_LABELS,
    nrow=NROW_ACC,
    figsize=ACC_FIGSIZE,
    save_path=OUTPUT_DIR / "selected_accuracy_cond2_sub230.svg",
)


Figure saved to /home/yangjiong/CategoryLearning_gitcode/results/selected_subject/selected_accuracy_cond2_sub230.svg


In [33]:
for sid, label in zip(SUBJECT_IDS, SUBJECT_LABELS):
    plot_oral_vs_model_fig3c_style(
        oral_hypo_hits,
        model_results,
        subject_id=sid,
        subject_label=label,
        figsize=K_FIGSIZE,
        save_path=OUTPUT_DIR / f"selected_oral_vs_model_sub{sid}.svg",
    )


Figure saved to /home/yangjiong/CategoryLearning_gitcode/results/selected_subject/selected_oral_vs_model_sub230.svg
